# TikTok TechJam 2026 — Transformer GPU Benchmark (Colab)

Runs the official baseline vs our `UserOptimizedTransformer` on a **free Colab GPU** and reports correctness + speedup.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator → GPU` (a T4 is fine), then run the cells top to bottom.

## 1. Verify the GPU
If this prints `cuda available: False`, you haven't enabled the GPU runtime yet (see above).

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('compute capability:', torch.cuda.get_device_capability(0))

## 2. Get the code

Colab already has a CUDA build of PyTorch (and Triton), so there's nothing to install.

**Option A — upload (simplest).** Run the cell and pick **`torch_transformer_benchmark.py`** and **`sweep.py`** from your machine. (Optional: also upload the `transformer_opt/` folder if you want to try the Triton fused-LayerNorm path.)

In [ ]:
from google.colab import files
uploaded = files.upload()   # select torch_transformer_benchmark.py and sweep.py
print('uploaded:', list(uploaded.keys()))

**Option B — clone from GitHub** (once you've pushed the repo). Uncomment and set your URL:

In [ ]:
# !git clone https://github.com/<you>/<repo>.git
# %cd <repo>

## 3. Sanity check: accuracy + one speedup
Runs the official script on its default config in fp16. Look for `failed=0/...` (correctness) and a `speedup` line.

In [ ]:
!python torch_transformer_benchmark.py --device cuda --dtype float16 \
    --batch-size 8 --seq-len 512 --d-model 512 --heads 8 --ffn-dim 2048 --layers 6

## 4. Full sweep — fp16
Correctness + speedup across shapes (small/large batch, seq, dim). Saves `results/sweep.json`.

In [ ]:
!python sweep.py --dtype float16 --out results/fp16.json

## 5. Sweep — bfloat16 (more numerically forgiving)

In [ ]:
!python sweep.py --dtype bfloat16 --out results/bf16.json

## 6. Stacking wins (optional)

`torch.compile` (auto-fused Triton kernels), the causal + padding path, and the opt-in hand-written Triton LayerNorm. The first `--compile` run is slow (it's compiling); the timing is measured after warmup.

In [ ]:
!python sweep.py --dtype float16 --compile --out results/fp16_compile.json

In [ ]:
!python sweep.py --dtype float16 --causal --padding-ratio 0.4 --out results/fp16_causal_pad.json

In [ ]:
# Requires the transformer_opt/ folder to be present (Option A optional upload).
!python sweep.py --dtype float16 --triton-ln --out results/fp16_tritonln.json

## 7. Show collected results
Prints the JSON tables and the geomean speedup per configuration. Copy these into `docs/TECH_REPORT.md`.

In [ ]:
import glob, json, statistics
for path in sorted(glob.glob('results/*.json')):
    with open(path) as fh:
        data = json.load(fh)
    rows = data['rows']
    speeds = [r['speedup'] for r in rows if r['optimized_ms'] > 0]
    gm = statistics.geometric_mean(speeds) if speeds else float('nan')
    ok = all(r['passed'] for r in rows)
    print(f"\n=== {path} | {data['gpu']} | {data['dtype']} | "
          f"compile={data['compile']} triton_ln={data['triton_ln']} "
          f"causal={data['causal']} pad={data['padding_ratio']} ===")
    print(f"correctness: {'ALL PASS' if ok else 'FAIL'} | geomean speedup: {gm:.2f}x")
    for r in rows:
        print(f"  {r['label']:<28} {r['speedup']:>6.2f}x  "
              f"(base {r['baseline_ms']:.3f} ms -> opt {r['optimized_ms']:.3f} ms)")

## 8. Download results (optional)

In [ ]:
from google.colab import files
import glob
for p in glob.glob('results/*.json'):
    files.download(p)